In [ ]:
from torch.utils.data import Dataset, DataLoader
import tiktoken
class GP2Dataset(Dataset):
    
    def __init__(self, text, tokenizer, max_length, stride):
        
        self.input_ids = []
        self.target_ids = []
        ids = tokenizer.encode(text)
        
        for i in range(0, len(ids) - max_length, stride):
            input = ids[i: i + max_length]
            target = ids[i + 1 : i + max_length + 1 ]
            self.input_ids.append(input)
            self.target_ids.append(target)
            
    def __len__(self):
        return len(self.input_ids)        
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]        
        
  


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

with open("../data/mytext.txt", "r", encoding="utf-8") as f:
    text = f.read()


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    
    
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Multi head attention

First, I define a self attention class
after that I implement casual attention class
finally, implement the multihead one

In [ ]:
input = torch.randint(0, 50257, (1,6))  #creating an input seq of 6 tokens and 3 embedded size
print(input)

In [ ]:
#first we create a word embedding and positional embedding so inputs ready for self attention
import torch.nn as nn

embed = nn.Embedding(50257, 3)
embeded = embed(input)

pos_embed = nn.Embedding(6, 3)
po = pos_embed(torch.arange(6))

inputs = embeded + po
print(inputs)

In [ ]:
#here I only considered the second input, and then finds the context vector for that.
inputs = inputs.squeeze(0)
print(inputs.shape)
query = inputs[1]
print(query.shape)
att_score_2 = torch.empty(inputs.shape[0])
print(att_score_2)    

for i, x_i in enumerate(inputs):
    att_score_2[i] = torch.dot(x_i, query)
    
att_score_2 = torch.softmax(att_score_2,dim=-1)
print(att_score_2) 

contex_vec = torch.zeros(query.shape)
for i, x in enumerate(inputs):
    
    contex_vec += att_score_2[i] * x
       
print(contex_vec)    

In [ ]:
#we can overgeneralize it by using matrix multiplication 
att_score = inputs @ inputs.T
print(att_score.shape)

att_weights = torch.softmax(att_score, dim=-1)
print(att_weights)
print("All row sums:", att_weights.sum(dim=-1))

context_vec = att_weights @ inputs
print(context_vec)


In [ ]:
#now we can move forward and use trainable weights
#divide by sqrt of dimension since we want to keep the variance close to 1, it has a relation with variance.
#Also we dont want the values to be large
class SelfAttention(nn.Module):
    
    def __init__(self, d_in, d_out, qkv_bias =False):
        super().__init__()
        self.W_key = nn.Linear(d_in, d_out)
        self.W_query = nn.Linear(d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out)
        
    def forward(self, x):
        keys = self.W_key(x)
        values = self.W_value(x)
        query = self.W_query(x)
        
        att_score = query @ keys.T
        att_weights = torch.softmax(att_score / keys.shape[-1] **0.5, dim=-1)
        print(keys.shape[-1])
        contex_vector = att_weights @ values
        
        return contex_vector            

torch.manual_seed(789)
sa_v2 = SelfAttention(3, 2)
print(sa_v2(inputs))



In [ ]:
mask = torch.triu(torch.ones(6,6), diagonal=1)
print(mask)
scores = torch.randn(6,6)
scores.masked_fill_(mask.bool(),-torch.inf)
print(scores)




In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
import torch.nn as nn

embed = nn.Embedding(50257, 3)
embeded = embed(input)

pos_embed = nn.Embedding(6, 3)
po = pos_embed(torch.arange(6))

inputs = embeded + po
print(inputs)
ca = CausalAttention(3,2,6,False)
contex = ca(inputs)
print(contex)

In [ ]:
x = torch.randn(2, 3)
print(x)
x.view(-1)
print(x)


In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [ ]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch)
print(batch.shape) 

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

In [3]:
import sys, os
sys.path.append("..")


In [3]:
from layers.transformer_block import TransformerBlock
from config.loader import CONFIG

config = CONFIG
print(config)

{'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}


In [4]:
import torch
torch.manual_seed(123)
x = torch.rand(2, 50, 768) #A
block = TransformerBlock(config)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 50, 768])
Output shape: torch.Size([2, 50, 768])


In [5]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

In [6]:
from model.gpt_model import GPTModel
torch.manual_seed(123)
model = GPTModel(config)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.7955e-01,  2.8515e-01, -7.6131e-01,  ..., -4.8374e-01,
          -4.2503e-01, -1.7187e-01],
         [-6.2581e-01, -3.7480e-01, -9.7020e-01,  ...,  1.9168e-01,
          -1.3234e+00, -2.7643e-01],
         [ 5.1744e-01,  1.3866e-01,  2.4886e-01,  ...,  3.5054e-01,
          -7.7531e-02, -8.0041e-02],
         [-2.5664e-01, -6.9693e-01, -9.9479e-01,  ..., -4.4732e-02,
           6.1773e-02,  1.3467e-01]],

        [[-2.2379e-01,  1.1651e-01, -9.9836e-01,  ..., -1.5730e-01,
          -4.4800e-01, -2.8646e-02],
         [-8.7227e-01, -3.9389e-01, -1.1099e+00,  ...,  3.3035e-01,
          -9.2395e-02, -1.8477e-05],
         [ 4.5988e-01, -1.4274e-01, -1.2227e-01,  ...,  2.7490e-01,
           5.8297e-02, -8.9930e-02],
         [-6.2163e-01, -4.4854e-01, -4.7675e-01,  ..., -3.6519e-01,
           3.4402e-01, -3.8154e-01]]], grad_fn=<UnsafeViewBackward0>)


In [1]:
sys.path.append("..")

from utils.tokenization import text_to_token_ids, token_ids_to_text


NameError: name 'sys' is not defined

In [9]:
token_ids = model.generate(
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    top_k=25,
    temperature=1.4
)

In [46]:
x = torch.tensor([0.2,0.5,0.3])
print(x)

tensor([0.2000, 0.5000, 0.3000])


In [52]:
b = torch.where(x<0.4, torch.tensor(float("-inf")), x)
b = torch.softmax(b, dim=-1)
c = torch.argmax(b)
print(b)
print(c)

tensor([0., 1., 0.])
tensor(1)


In [10]:
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves youattiquad investigatingfellorians Mel POWER Jin flushed Thurs groundwater MUSToppy Epicodi


In [5]:
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)


cuda


In [8]:
import sys
print(sys.executable)


d:\projects\GPT2_from_scratch\venv\Scripts\python.exe


In [3]:
from config.loader import CONFIG

In [4]:
config_model = CONFIG['model']
config_training = CONFIG["training"]

In [5]:
print(config_model)
print(config_training)

{'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}
{'batch_size': 2, 'num_epochs': 10, 'lr': 0.0004, 'weight_decay': 0.1, 'eval_freq': 5, 'eval_iter': 5, 'seed': 123}


In [1]:
import sys, os
sys.path.append("..")


In [2]:
from utils.gpt2.load_gpt2_tf_weights import download_gpt2
settings, params = download_gpt2(model_size="124M")

d:\projects\GPT2_from_scratch\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
models\124M\checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 21.2kiB/s]
d:\projects\GPT2_from_scratch\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
models\124M\encoder.json: 100%|██████████| 1.04M/1.04M [00:02<00:00, 517kiB/s]
d:\projects\GPT2_from_scratch\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'op

In [3]:
from utils.gpt2.load_weights import load_weights_into_gpt
from model.gpt_model import GPTModel
from config.loader import CONFIG
gpt = GPTModel(CONFIG["model"])


In [6]:
gpt.eval()
load_weights_into_gpt(gpt, params)
gpt.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm()
  (out_head): Linear(in_features=768, out_features=50257, bias=Fa

In [7]:
config = CONFIG["model"]

In [14]:
from utils.tokenization import text_to_token_ids,token_ids_to_text
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [19]:
token_ids = gpt.generate(
    
    idx=text_to_token_ids("I want to know how to install gpt2?", tokenizer).to(device),
    max_new_tokens=100,
    top_k=5,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 I want to know how to install gpt2? I want a simple install that does the job and is easy to follow.

Step 5 – Install Gpt2 from Source

Gpt2 is the default install tool for Linux. You can find the source of it at: http://github.com/gctools/gpt2/tree/git/gpt2

The Gpt2 package contains two commands.

The first command is a simple command that will install Gpt2 and run the Gpt


In [2]:
from train.train import main
main()

cuda
Ep 1 (Step 000000): Train loss 9.819, Val loss 9.926
Ep 1 (Step 000005): Train loss 8.070, Val loss 8.341
Every effort moves you,,,,,,,,,,,,.                                     
Ep 2 (Step 000010): Train loss 6.624, Val loss 7.051
Ep 2 (Step 000015): Train loss 6.047, Val loss 6.599
Every effort moves you, and,, and,, and,,,, and,.                                   
Ep 3 (Step 000020): Train loss 5.567, Val loss 6.483
Ep 3 (Step 000025): Train loss 5.507, Val loss 6.408
Every effort moves you, and, and of the of the of the, and, and. G. Gis, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and,
Ep 4 (Step 000030): Train loss 5.090, Val loss 6.324
Ep 4 (Step 000035): Train loss 4.862, Val loss 6.334
Every effort moves you.  "I had been the picture-- the picture.               "I was a the of the of the of the of the of the picture"I had been the of
Ep 5 (Step 000040): Train loss 4.262, Val loss 6.217
Every effort moves you know the "I had been--I to me--as of 

In [ ]:
def clac_loss(logits,target,model):
    loss = torch.nn.functional.cross_entropy(logit.flatten(0,1), target.flatten())